In [ ]:
!git clone https://github.com/MTG/mtg-jamendo-dataset.git
%cd mtg-jamendo-dataset
!pip install -r scripts/requirements.txt

In [ ]:
show(n)

In [ ]:
!head -3 data/download/raw_30s_audio-low_sha256_tars.txt


In [ ]:
!mkdir -p /content/audio
%cd /content/audio
for f in ["raw_30s_audio-low-00.tar", "raw_30s_audio-low-01.tar"]:
    !wget -q --show-progress https://cdn.freesound.org/mtg-jamendo/raw_30s/audio-low/{f}
!sha256sum raw_30s_audio-low-00.tar raw_30s_audio-low-01.tar
!for f in *.tar; do tar -xf $f && rm $f; done
!ls | head; find . -name "*.mp3" | wc -l

In [ ]:
!tar -xf raw_30s_audio-low-00.tar && rm raw_30s_audio-low-00.tar
!ls; find . -name "*.mp3" | wc -l

In [ ]:
import csv, glob, os

tags = {}
with open('/content/mtg-jamendo-dataset/data/raw_30s_cleantags.tsv') as f:
    next(f)
    for row in csv.reader(f, delimiter='\t'):
        tid = int(row[0].split('_')[1])
        tags[tid] = row[5:]

files = glob.glob('/content/audio/*/*.mp3')
tid_of = lambda p: int(os.path.basename(p).split('.')[0])
instrumental = [p for p in files if 'instrument---voice' not in tags.get(tid_of(p), [])]
print(len(files), len(instrumental))

In [ ]:
!rm -rf /content/audio
!mkdir -p /content/audio
!tar -xf /content/drive/MyDrive/sonar/tars/raw_30s_audio-low-00.tar -C /content/audio
!find /content/audio -name "*.mp3" | wc -l

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -q https://github.com/MTG/mtg-jamendo-dataset.git /content/mtg-jamendo-dataset
!mkdir -p /content/audio
!tar -xf /content/drive/MyDrive/sonar/tars/raw_30s_audio-low-00.tar -C /content/audio
!tar -xf /content/drive/MyDrive/sonar/tars/raw_30s_audio-low-01.tar -C /content/audio
!find /content/audio -name "*.mp3" | wc -l

In [ ]:
!git clone -q https://github.com/MTG/mtg-jamendo-dataset.git /content/mtg-jamendo-dataset

In [ ]:
!ls -lh /content/drive/MyDrive/sonar/tars
!find /content/audio -name "*.mp3" | wc -l

In [ ]:
!tar -xf /content/drive/MyDrive/sonar/tars/raw_30s_audio-low-00.tar -C /content/audio
!tar -xf /content/drive/MyDrive/sonar/tars/raw_30s_audio-low-01.tar -C /content/audio
!find /content/audio -name "*.mp3" | wc -l

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/sonar

In [ ]:
import torch, librosa, numpy as np
from transformers import ClapModel, ClapProcessor

name = 'laion/larger_clap_music_and_speech'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = ClapModel.from_pretrained(name).to(device).eval()
proc = ClapProcessor.from_pretrained(name)

def chunk_audio(waveform, sr=48000, secs=10):
    chunk_len = sr * secs
    chunks = [waveform[s:s+chunk_len] for s in range(0, len(waveform), chunk_len)]
    return [c for c in chunks if len(c) >= chunk_len // 2]   # drop tiny leftover tail

embs, ids = [], []
for p in instrumental:
    y = librosa.load(p, sr=48000, duration=30)[0]
    chunks = chunk_audio(y)
    if not chunks: chunks = [y]
    inputs = proc(audio=chunks, sampling_rate=48000, return_tensors='pt').to(device)
    with torch.no_grad():
        e = model.get_audio_features(**inputs)
        e = e if torch.is_tensor(e) else e.pooler_output
    e = torch.nn.functional.normalize(e.mean(dim=0), dim=-1)
    embs.append(e.cpu().numpy())
    ids.append(tid_of(p))

embs = np.stack(embs)
np.save('/content/drive/MyDrive/sonar/audio_embs_run002.npy', embs)
np.save('/content/drive/MyDrive/sonar/ids_run002.npy', np.array(ids))
print(embs.shape)

In [ ]:
!ls /content/audio; find /content/audio -name "*.mp3" | wc -l

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
D = '/content/drive/MyDrive/sonar/tars'
os.makedirs(D, exist_ok=True)
!git clone -q https://github.com/MTG/mtg-jamendo-dataset.git /content/mtg-jamendo-dataset

for f in ["raw_30s_audio-low-00.tar", "raw_30s_audio-low-01.tar"]:
    if not os.path.exists(f"{D}/{f}"):
        !wget -q --show-progress -O {D}/{f} https://cdn.freesound.org/mtg-jamendo/raw_30s/audio-low/{f}
    !mkdir -p /content/audio && tar -xf {D}/{f} -C /content/audio

!find /content/audio -name "*.mp3" | wc -l

In [ ]:
import json, time
queries = ["dark trap beat with heavy 808s", "nighttime jazz piano beat",
  "energetic electronic beat with punchy drums", "sad lo-fi hip hop with soft guitar",
  "chill relaxing beat with smooth synth pads", "epic cinematic beat with strings",
  "happy upbeat funk beat with bass guitar", "dreamy ambient beat with reverb piano",
  "aggressive rock beat with distorted guitar", "romantic slow beat with acoustic guitar"]

path_of = {tid_of(p): p for p in instrumental}
run = {"model": name, "clip_seconds": 10, "results": []}
for qid, q in enumerate(queries, 1):
    t0 = time.time()
    with torch.no_grad():
        t = model.get_text_features(**proc(text=[q], return_tensors='pt', padding=True).to(device))
        t = t if torch.is_tensor(t) else t.pooler_output
    t = torch.nn.functional.normalize(t, dim=-1).cpu().numpy()[0]
    scores = embs @ t                                  # cosine similarity (vectors are unit length)
    top = scores.argsort()[::-1][:10]
    ms = round((time.time() - t0) * 1000)
    run["results"].append({"id": qid, "prompt": q, "latency_ms": ms,
        "top10": [{"track": int(ids[i]), "score": float(scores[i])} for i in top]})
    print(qid, ms, "ms", q)

json.dump(run, open('/content/drive/MyDrive/sonar/run_001.json', 'w'), indent=1)

In [ ]:
from IPython.display import Audio, display
def show(qid):
    r = run["results"][qid-1]
    print("PROMPT:", r["prompt"])
    for rank, hit in enumerate(r["top10"], 1):
        y, sr = librosa.load(path_of[hit["track"]], sr=22050, duration=15)
        print(rank, [t.split('---')[1] for t in tags.get(hit["track"], [])])
        display(Audio(y, rate=sr))
show(1)

In [ ]:
labels = {1: [1, 4]}
json.dump(labels, open('/content/drive/MyDrive/sonar/labels.json', 'w'))
show(2)

In [ ]:
labels[2] = [3, 5, 7, 9]
json.dump(labels, open('/content/drive/MyDrive/sonar/labels.json', 'w'))
show(3)

In [ ]:
labels[3] = [1, 2, 3, 4, 7, 9]
json.dump(labels, open('/content/drive/MyDrive/sonar/labels.json', 'w'))
show(4)

In [ ]:
labels[4] = []
json.dump(labels, open('/content/drive/MyDrive/sonar/labels.json', 'w'))
show(5)

In [ ]:
labels[5] = [1, 5, 7, 8, 9, 10]
json.dump(labels, open('/content/drive/MyDrive/sonar/labels.json', 'w'))
show(6)

In [ ]:
labels[6] = [1, 2, 3, 4]
json.dump(labels, open('/content/drive/MyDrive/sonar/labels.json', 'w'))
show(7)

In [ ]:
labels[7] = [6, 7, 8]
json.dump(labels, open('/content/drive/MyDrive/sonar/labels.json', 'w'))
show(8)

In [ ]:
labels[8] = [3, 8, 10]
json.dump(labels, open('/content/drive/MyDrive/sonar/labels.json', 'w'))
show(9)

In [ ]:
labels[9] = [1, 7, 9]
json.dump(labels, open('/content/drive/MyDrive/sonar/labels.json', 'w'))
show(10)

In [ ]:
labels[10] = [5, 10]
json.dump(labels, open('/content/drive/MyDrive/sonar/labels.json', 'w'))

import numpy as np
def ndcg10(ranks):
    if not ranks: return 0.0
    dcg = sum(1/np.log2(r+1) for r in ranks)
    idcg = sum(1/np.log2(i+2) for i in range(len(ranks)))
    return dcg / idcg

scores = {q: round(ndcg10(r), 2) for q, r in labels.items()}
print(scores, "mean nDCG@10 =", round(np.mean(list(scores.values())), 2))

In [ ]:
json.dump({"run": "002", "scores": {k: float(v) for k, v in scores.items()},
           "mean_ndcg10": 0.64},
          open('/content/drive/MyDrive/sonar/eval_run002.json', 'w'), indent=1)

In [ ]:
labels = {1: [4], 2: [4]}
json.dump(labels, open('/content/drive/MyDrive/sonar/labels.json', 'w'))
print(labels)

In [ ]:
show(3)

In [ ]:
from IPython.display import Audio, display
def show(qid):
    r = run["results"][qid-1]
    print("PROMPT:", r["prompt"])
    for rank, hit in enumerate(r["top10"], 1):
        y, sr = librosa.load(path_of[hit["track"]], sr=22050, duration=15)
        print(rank, [t.split('---')[1] for t in tags.get(hit["track"], [])])
        display(Audio(y, rate=sr))

In [ ]:
labels[3] = []
json.dump(labels, open('/content/drive/MyDrive/sonar/labels.json', 'w'))
show(4)

In [ ]:
import numpy as np
def ndcg10(ranks):
    if not ranks: return 0.0
    dcg = sum(1/np.log2(r+1) for r in ranks)
    idcg = sum(1/np.log2(i+2) for i in range(len(ranks)))
    return dcg / idcg

scores = {q: round(ndcg10(r), 2) for q, r in labels.items()}
print(scores, "mean nDCG@10 =", round(np.mean(list(scores.values())), 2))

In [ ]:
notepad docs\scoping.md

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -q https://github.com/MTG/mtg-jamendo-dataset.git /content/mtg-jamendo-dataset
!mkdir -p /content/audio
!tar -xf /content/drive/MyDrive/sonar/tars/raw_30s_audio-low-00.tar -C /content/audio
!tar -xf /content/drive/MyDrive/sonar/tars/raw_30s_audio-low-01.tar -C /content/audio
!find /content/audio -name "*.mp3" | wc -l